In [1]:
import asyncio
import os
from dotenv import load_dotenv
import aiohttp

# Корректно и жестко определяем корень проекта для Windows среды Jupyter
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

# Загружаем переменные окружения
dotenv_path = os.path.join(project_root, 'scripts', '.env')
load_dotenv(dotenv_path=dotenv_path)
WB_TOKEN = os.getenv("WB_TOKEN_ANALYTICS")

async def test_wb_connection():
    if not WB_TOKEN:
        print("Ошибка: Переменная WB_TOKEN не найдена в .env!")
        print(f"Попытка прочитать файл по пути: {os.path.abspath(dotenv_path)}")
        return

    # GET-метод возвращает список ранее созданных задач генерации отчетов
    url = "https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads"
    
    headers = {
        "Authorization": WB_TOKEN,
        "accept": "application/json"
    }
    
    async with aiohttp.ClientSession() as session:
        try:
            # Используем GET для безопасной проверки связи и получения списка
            async with session.get(url, headers=headers) as response:
                print(f"Статус HTTP-ответа: {response.status}")
                
                if response.status == 200:
                    data = await response.json()
                    print("Успешно подключено к API Аналитики!")
                    # Выводим весь ответ, чтобы изучить ключи (data, error, meta и т.д.)
                    print("Полная структура ответа от WB:")
                    print(data)
                elif response.status == 401:
                    print("Ошибка 401: Неверный токен аналитики.")
                else:
                    text = await response.text()
                    print(f"Ответ сервера (Код {response.status}): {text}")
        except Exception as e:
            print(f"Произошла ошибка при обработке данных: {e}")

await test_wb_connection()


Статус HTTP-ответа: 200
Успешно подключено к API Аналитики!
Полная структура ответа от WB:
{'data': []}


In [2]:
import asyncio
import os
from dotenv import load_dotenv
import aiohttp

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

dotenv_path = os.path.join(project_root, 'scripts', '.env')
load_dotenv(dotenv_path=dotenv_path)

# Токен категории 'Финансы'
WB_FINANCE_TOKEN = os.getenv("WB_TOKEN_FINANCE") or os.getenv("WB_FINANCE_TOKEN")

async def get_financial_reports_list():
    if not WB_FINANCE_TOKEN:
        print("Ошибка: Токен категории 'Финансы' не найден в файле .env!")
        return

    # Эндпоинт получения списка отчетов реализации
    url = "https://finance-api.wildberries.ru/api/finance/v1/sales-reports/list"
    
    headers = {
        "Authorization": WB_FINANCE_TOKEN,
        "accept": "application/json",
        "Content-Type": "application/json"
    }
    
    # Ищем еженедельные отчеты за последний месяц
    payload = {
        "dateFrom": "2026-09-20",
        "dateTo": "2026-10-22",
        "limit": 10,
        "offset": 0,
        "period": "weekly"
    }
    
    async with aiohttp.ClientSession() as session:
        try:
            print("Запрос списка финансовых отчетов...")
            async with session.post(url, headers=headers, json=payload) as response:
                print(f"Статус HTTP-ответа: {response.status}")
                
                if response.status == 200:
                    data = await response.json()
                    print("Список отчетов успешно получен!")
                    print(f"Найдено отчетов в ответе: {len(data)}")
                    if data:
                        print("Данные первого найденного отчета (для проверки):")
                        print(data[0])
                elif response.status == 204:
                    print("Статус 204: За указанный период финансовые отчеты отсутствуют.")
                else:
                    text = await response.text()
                    print(f"Ошибка {response.status}: {text}")
        except Exception as e:
            print(f"Ошибка выполнения запроса: {e}")

await get_financial_reports_list()


Запрос списка финансовых отчетов...
Статус HTTP-ответа: 403
Ошибка 403: {"status":403,"statusText":"Forbidden","title":"Forbidden","detail":"scope is not allowed for this resource","requestId":"b2bbc61205740ee62485bf7a49cf9759","origin":"ag-finance","timestamp":"2026-10-23T09:01:44Z"}
